In [1]:
import pandas as pd 
from pyspark.sql import SparkSession
from pathlib import Path
import os
import glob 
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql import types as T


In [2]:
spark = SparkSession.builder\
        .appName("pfmsbenreq")\
        .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/08/31 08:01:26 WARN Utils: Your hostname, 2640L, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/08/31 08:01:26 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/yogavarman/venvs/global_env/lib/python3.14/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/08/31 08:01:27 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
filepath = "/mnt/d/all_data/20260831/need to push pfms registration list.csv"

import os

print("Exists:", os.path.isfile(filepath))
print("Path:", filepath)

Exists: False
Path: /mnt/d/all_data/20260831/need to push pfms registration list.csv


In [4]:
import glob

files = glob.glob("/mnt/d/all_data/20260831/*")

print(files)

['/mnt/d/all_data/20260831/202608311326.csv', '/mnt/d/all_data/20260831/MAD_[2026-08-13]_202608311055.csv', '/mnt/d/all_data/20260831/MAS_[2026-08-13]_202608311053.csv', '/mnt/d/all_data/20260831/MA_[2026-08-13]_202608311055.csv', '/mnt/d/all_data/20260831/NDFA_[2026-08-10]_202608311101.csv', '/mnt/d/all_data/20260831/need to push pfms registration list']


In [33]:
filepath = "/mnt/d/all_data/20260831/need to push pfms registration list"

df = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .option("quote", '"')
    .option("escape", '"')
    .option("multiLine", True)
    .csv(filepath)
)

In [34]:
df.show()

+------------+-----------------+----------+--------------------+---------+------+------------+---------+------------+-------+----------+------------------+----------------------+-----------------------+-----------------+--------------+---------+------------------+------------+-----------+--------------+-----------+------+-----------+--------+--------------------+----------------+--------+--------------------+-------------------------+----------+------------------+--------------------+----------+----------+------+--------+--------+---------+--------------------+----------------+------------------------+---------------+-----------------------+-------------------------+-------------------+--------------------+---------------+-------------+---------------+-----------------+--------------+---------+-------+-------+---------+---------------------+------------------------+---------------------------+--------------------+--------------+------------------+-------------------------+-------------

26/08/31 08:57:22 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: aadhaar_no, education_type_id, student_id, student_name, gender_id, gender, community_id, community, aadhaar_no, umis_no, emis_no, isdifferentlyabled, disability_category_id, student_registration_no, accademic_year_id, institution_id, course_id, academic_status_id, studing_year, is_hosteler, hostel_type_id, hostel_type, medium, religion_id, religion, student_education_id, aadhaar_verified, state_id, student_updatedts, studenteduction_updatedts, mobile_no, mobile_no_verified, disability_name, mediumname, state_name, income, incomeid, is_rural, education, institution_name, institution_code, institution_ownership_id, ownership, institution_category_id, institution_category_name, institution_type_id, institute_type, district_name, district_code, districtlgdcode, taluk_name, taluk_lgd_code, taluk_lgd, address, pincode, scheme_id, sch_accademic_year_id, sch_student_education_id, in_eligible_rule_cond

In [35]:
df = df.select(
    "aadhaar_no0","application_id","student_name","districtlgdcode","district_name","scheme_name","pfms_scheme_code","gender")

In [36]:
df = (
    df
    .withColumnRenamed("student_name", "beneficiary_name")
    .withColumnRenamed("aadhaar_no0", "aadhaar_number")
    .withColumnRenamed("district_name", "addressline_1")
    .withColumnRenamed("districtlgdcode", "district_lgt_code")
    .withColumn("scheme_id", F.lit(4))
    .withColumn("scheme_name_identifier",F.lit(10))
    .withColumn("purpose", F.lit("A"))
    .withColumn("state_lgt_code", F.lit(33))
    .withColumn(
        "gender",
        F.when(
            F.lower(F.trim(F.col("gender"))) == "female",
            F.lit("F")
        ).otherwise(F.lit("M"))
    )
    .withColumn("reference_id",F.lit("man202608311"))
)

In [37]:
df=df.drop("pfms_scheme_code")
df.show()

+--------------+--------------------+--------------------+-----------------+---------------+--------------------+------+---------+----------------------+-------+--------------+------------+
|aadhaar_number|      application_id|    beneficiary_name|district_lgt_code|  addressline_1|         scheme_name|gender|scheme_id|scheme_name_identifier|purpose|state_lgt_code|reference_id|
+--------------+--------------------+--------------------+-----------------+---------------+--------------------+------+---------+----------------------+-------+--------------+------------+
|  470653094055|TNISSPEMIS2026001...|            JESIKA N|              579|   NAGAPATTINAM|ADW GOI PRE MATRI...|     F|        4|                    10|      A|            33|man202608311|
|  297345503725|TNISSPEMIS2026001...|          RAGAVI R J|              581|     PERAMBALUR|ADW GOI PRE MATRI...|     F|        4|                    10|      A|            33|man202608311|
|  748443637115|TNISSPEMIS2026001...|             

26/08/31 08:59:21 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: aadhaar_no, education_type_id, student_id, student_name, gender_id, gender, community_id, community, aadhaar_no, umis_no, emis_no, isdifferentlyabled, disability_category_id, student_registration_no, accademic_year_id, institution_id, course_id, academic_status_id, studing_year, is_hosteler, hostel_type_id, hostel_type, medium, religion_id, religion, student_education_id, aadhaar_verified, state_id, student_updatedts, studenteduction_updatedts, mobile_no, mobile_no_verified, disability_name, mediumname, state_name, income, incomeid, is_rural, education, institution_name, institution_code, institution_ownership_id, ownership, institution_category_id, institution_category_name, institution_type_id, institute_type, district_name, district_code, districtlgdcode, taluk_name, taluk_lgd_code, taluk_lgd, address, pincode, scheme_id, sch_accademic_year_id, sch_student_education_id, in_eligible_rule_cond

In [38]:
pdf = df.toPandas()
output_path = "/mnt/d/all_data/20260831/pfms_registration_list.csv"

pdf.to_csv(
    output_path,
    index=False
)

print(f"CSV created: {output_path}")

CSV created: /mnt/d/all_data/20260831/pfms_registration_list.csv


/home/yogavarman/venvs/global_env/lib/python3.14/site-packages/pyspark/sql/pandas/conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/08/31 08:59:22 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: aadhaar_no, education_type_id, student_id, student_name, gender_id, gender, community_id, community, aadhaar_no, umis_no, emis_no, isdifferentlyabled, disability_category_id, student_registration_no, accademic_year_id, institution_id, course_id, academic_status_id, studing_year, is_hosteler, hostel_type_id, hostel_type, medium, religion_id, religion, student_education_id, aadhaar_verified, state_id, student_updatedts, studenteduction_updatedts, mobile_no, mobile_no_verified, disability_name, mediumname, state_name, income, incomeid, is_rural, education, institution_name, institution_code, institution_ownership